In [1]:
import os
os.environ["TRITON_INTERPRET"] = "1"

In [11]:
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/MLProject/Code')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!pwd


/content/drive/MyDrive/MLProject/Code


In [13]:
#Cuda profiler?
import time
import nvtx

In [14]:
!nvidia-smi
!nvcc --version
!wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb
!apt update
!apt install ./nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb
!apt --fix-broken install
!nsys --version

Mon Mar 30 18:28:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Unfused GEMM and ReLU

Used the unfused triton version - to profile an arbritrary CNN

In [15]:
%%writefile nsys_unfused.py

import torch
import torch.nn as nn
import torch.optim as optim
import triton
import triton.language as tl
from triton_unfused_kernel import unfused_gemm_relu
device = torch.device("cuda")
class MyCustomCNN_unfused(nn.Module):
    def __init__(self, in_ch, out_fil, k_size, num_cl, in_res):
        super().__init__()
        self.k_size = k_size
        self.out_fil = out_fil
        self.weights = nn.Parameter((torch.randn(out_fil, in_ch, k_size, k_size)))
        self.bias = nn.Parameter(torch.zeros(out_fil))

        self.unfold = nn.Unfold(kernel_size=k_size)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

      # Dynamic calculation for the linear layer
        conf_out_dim = (in_res - k_size) + 1
        pool_out_dim = conf_out_dim // 2
        self.flattened_size = out_fil * pool_out_dim * pool_out_dim

        self.fc = nn.Linear(self.flattened_size, num_cl)

    def forward(self, x):
        b_size = x.shape[0]

        x_unfolded = self.unfold(x)
        w_flat = self.weights.view(self.out_fil, -1)
        torch.cuda.nvtx.range_push("unfused_gemm_relu")
        x_conv = unfused_gemm_relu(w_flat, x_unfolded[0], self.bias.view(1, -1, 1))
        torch.cuda.synchronize()
        torch.cuda.nvtx.range_pop()
        out_h = x.shape[2] - self.k_size + 1
        out_w = x.shape[3] - self.k_size + 1
        x = x_conv.view(b_size, self.out_fil, out_h, out_w)

        #x = self.relu(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)

        logits = self.fc(x)

        return logits

model = MyCustomCNN_unfused(in_ch=3, out_fil=16, k_size=3, num_cl=2, in_res = 128).to(device)
input_data = torch.randn(1, 3, 128, 128).to(device) # Batch of 2, RGB, 8x8

#warmup
#op_wu = model(input_data)

# Final run for profiling
output = model(input_data)
print(f"Success! Output shape: {output.shape} on GPU")

Overwriting nsys_unfused.py


In [16]:
t0 = time.time()
!nsys profile -f true -o nsys_unfused python nsys_unfused.py
t1 = time.time()
print("time taken: ", (t1 - t0))

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/triton/runtime/interpreter.py", line 1357, in __call__
    self.fn(**args)
  File "/content/drive/MyDrive/MLProject/Code/triton_unfused_kernel.py", line 19, in gemm_kernel
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
                               ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/triton/runtime/interpreter.py", line 859, in <lambda>
    new_member = lambda *args, member=member, **kwargs: (member(*args, **
                                                         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/triton/language/core.py", line 43, in wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/triton/language/core.py", line 1644, in arange
    return _semantic.arange(start, end)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/triton/lan

# Fused GEMM and ReLU

In [18]:

%%writefile nsys_fused.py

import torch
import torch.nn as nn
import torch.optim as optim
import triton
import triton.language as tl
from triton_fused_kernel import fused_gemm_relu

class MyCustomCNN_fused(nn.Module):
    def __init__(self, in_ch, out_fil, k_size, num_cl, in_res):
        super().__init__()
        self.k_size = k_size
        self.out_fil = out_fil
        self.weights = nn.Parameter(torch.randn(out_fil, in_ch, k_size, k_size))
        self.bias = nn.Parameter(torch.randn(out_fil))

        self.unfold = nn.Unfold(kernel_size=k_size)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Dynamic calculation for the linear layer
        conf_out_dim = (in_res - k_size) + 1
        pool_out_dim = conf_out_dim // 2
        self.flattened_size = out_fil * pool_out_dim * pool_out_dim

        self.fc = nn.Linear(self.flattened_size, num_cl)

    def forward(self, x):
        b_size = x.shape[0]

        x_unfolded = self.unfold(x)
        w_flat = self.weights.view(self.out_fil, -1)
        torch.cuda.nvtx.range_push("fused_gemm_relu")
        x_conv = fused_gemm_relu(w_flat, x_unfolded[0], self.bias.view(1, -1, 1))
        torch.cuda.synchronize()
        torch.cuda.nvtx.range_pop()
        out_h = x.shape[2] - self.k_size + 1
        out_w = x.shape[3] - self.k_size + 1
        x = x_conv.view(b_size, self.out_fil, out_h, out_w)

        #x = self.relu(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)

        logits = self.fc(x)

        return logits

model = MyCustomCNN_fused(in_ch=3, out_fil=16, k_size=3, num_cl=2, in_res=512)
input_data = torch.randn(1, 3, 512, 512) # Batch of 2, RGB, 8x8
#warmup
op_wu = model(input_data)

# Final run for profiling
output = model(input_data)
print(f"Success! Output shape: {output.shape} on GPU")

Overwriting nsys_fused.py


In [19]:
t0 = time.time()
!nsys profile -f true -o nsys_fused python nsys_fused.py
t1 = time.time()
print("time taken: ", (t1 - t0))

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/triton/runtime/interpreter.py", line 1357, in __call__
    self.fn(**args)
  File "/content/drive/MyDrive/MLProject/Code/triton_fused_kernel.py", line 20, in fused_gemm_relu_kernel
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
                               ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/triton/runtime/interpreter.py", line 859, in <lambda>
    new_member = lambda *args, member=member, **kwargs: (member(*args, **
                                                         ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/triton/language/core.py", line 43, in wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/triton/language/core.py", line 1644, in arange
    return _semantic.arange(start, end)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/t

# Native implementation


In [20]:
%%writefile nsys_native.py

import torch
import torch.nn as nn
import torch.optim as optim
#import triton
#import triton.language as tl
#from triton_fused_kernel import fused_gemm_relu

class MyCustomCNN_native(nn.Module):
    def __init__(self, in_ch, out_fil, k_size, num_cl, in_res):
        super().__init__()
        self.k_size = k_size
        self.out_fil = out_fil
        self.weights = nn.Parameter(torch.randn(out_fil, in_ch, k_size, k_size))
        self.bias = nn.Parameter(torch.randn(out_fil))
        self.conv2d = nn.Conv2d(in_channels=in_ch, out_channels=out_fil, kernel_size=k_size)
        self.unfold = nn.Unfold(kernel_size=k_size)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Dynamic calculation for the linear layer
        conf_out_dim = (in_res - k_size) + 1
        pool_out_dim = conf_out_dim // 2
        self.flattened_size = out_fil * pool_out_dim * pool_out_dim

        self.fc = nn.Linear(self.flattened_size, num_cl)

    def forward(self, x):
        b_size = x.shape[0]


        torch.cuda.nvtx.range_push("native")
        x = self.conv2d(x)
        x = self.relu(x)
        torch.cuda.synchronize()
        torch.cuda.nvtx.range_pop()

        x = self.pool(x)
        x = torch.flatten(x, 1)

        logits = self.fc(x)

        return logits

model = MyCustomCNN_native(in_ch=3, out_fil=16, k_size=3, num_cl=2, in_res=512)
input_data = torch.randn(1, 3, 512, 512) # Batch of 2, RGB

#warmup
op_wu = model(input_data)

# Final run for profiling
output = model(input_data)
print(f"Success! Output shape: {output.shape} on GPU")

Overwriting nsys_native.py


In [21]:
t0 = time.time()
!nsys profile -f true -o nsys_native python nsys_native.py
t1 = time.time()
print("time taken: ", (t1 - t0))

Success! Output shape: torch.Size([1, 2]) on GPU
Generating '/tmp/nsys-report-e57b.qdstrm'
[1/1] [========================100%] nsys_native.nsys-rep
Generated:
    /content/drive/MyDrive/MLProject/Code/nsys_native.nsys-rep
time taken:  13.90967321395874
